In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from Base_Modules.Environments import Prison
from Prison_Strategies.Basic_Strategies import *
from Base_Modules.Game_Master import Game_Master
from Base_Modules.Action import Action_History, Prison_Actions, Duel_Matrix
import pandas as pd
from collections import defaultdict

In [3]:
prison = Prison()
actions = prison.Get_Actions()

In [4]:
strategies_list = [
    Random_Strategy(actions=actions),
    Random_Strategy(actions=actions, p_coop=0.1),
    Always_Betray(actions=actions),
    Always_Cooperate(actions=actions),
    Patient_Unforgiving(actions=actions),
    Copycat(actions=actions),
    Periodic(actions=actions, period=1),
]

number_of_strategies = len(strategies_list)

In [5]:
def Get_Index_By_Name(strategies : dict[int, Strategy], name : str) -> int:
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s) == name:
            return id
    for ix, (id, s) in enumerate(strategies.items()):
        if str(s).startswith(name):
            return id
    return -1

In [6]:
strategies = {}

for (i, s) in enumerate(strategies_list):
    strategies[i] = s
    s.Set_ID(i)

In [7]:
number_of_games = 1000
total_games_explicit = True
max_action_memory = -1

gm = Game_Master(prison, strategies=strategies, duel_size=2, max_action_memory=max_action_memory)
duel_matrix, rewards = gm.Tournament(number_of_games, Game_Master.Game_Type.All_Vs_All, total_games_explicit=total_games_explicit)
rewards.Sort_Total_Rewards()

## Analiza


In [8]:
def Sort_Based_On_Total_Rewards(total_rewards, data):
    sorted_data = dict(sorted(
        data.items(),
        key=lambda kv: total_rewards.get(kv[0], 0),
        reverse=True
    ))
    return sorted_data

In [9]:
total_rewards = rewards.Get_Total_Rewards()
total_rewards_per_name = {str(strategies[i]):total_rewards[i] for i in total_rewards.keys()}

average_rewards_per_match = {k: (float(v)/number_of_strategies) for k, v in total_rewards_per_name.items()}
average_rewards_per_round = {k: (float(v)/(number_of_strategies*number_of_games)) for k, v in total_rewards_per_name.items()}

duel_rewards = rewards.Get_All_Duel_Rewards()

duel_rewards = dict(sorted(duel_rewards.items(), key=lambda kv: total_rewards.get(kv[0][0]), reverse=True))

In [10]:
largest_shared_victory = -float("inf")
least_shared_victory = float("inf")

for (_, rewards) in duel_rewards.items():
    largest_shared_victory = max(largest_shared_victory, sum(rewards.values()))
    least_shared_victory = min(least_shared_victory, sum(rewards.values()))


print(f"Largest shared reward: {largest_shared_victory}")
print(f"Least shared reward: {least_shared_victory}")


Largest shared reward: 6000
Least shared reward: 2003


In [11]:
from Base_Modules.Nemesis import Nemesis_Best_Enemy_Score, Nemesis_Worst_Score, Nemesis_Largest_Difference

criterion = Nemesis_Worst_Score

nemesis = criterion.Get_Nemesis(duel_rewards=duel_rewards)
nemesis = Sort_Based_On_Total_Rewards(total_rewards=total_rewards, data=nemesis)
nemesis_per_name = criterion.Translate_Nemesis_To_Strategy_Names(strategies=strategies, nemesis=nemesis)

nemesis_df = pd.DataFrame(
    [(k, v[0]) for k, v in nemesis_per_name.items()],
    columns=["Strategy", "Its nemesis"]
)

display(nemesis_df)

,Strategy,Its nemesis
0,(2):Always_Betray,(4):Patient_Unforgiving (patience=1)
1,(4):Patient_Unforgiving (patience=1),(2):Always_Betray
2,(1):Random_Strategy (p_coop=0.1),(2):Always_Betray
3,(5):Copycat (1st:Cooperate),(2):Always_Betray
4,(6):Periodic (period=1),(2):Always_Betray
5,(0):Random_Strategy (p_coop=0.5),(4):Patient_Unforgiving (patience=1)
6,(3):Always_Cooperate,(2):Always_Betray


In [12]:
from Base_Modules.Nemesis import Friend_Best_Total_Score

criterion = Friend_Best_Total_Score

friends = criterion.Get_Nemesis(duel_rewards=duel_rewards)
friends = Sort_Based_On_Total_Rewards(total_rewards=total_rewards, data=friends)
friends_per_name = criterion.Translate_Nemesis_To_Strategy_Names(strategies=strategies, nemesis=friends)

friends_df = pd.DataFrame(
    [(k, v[0]) for k, v in friends_per_name.items()],
    columns=["Strategy", "Its friend"]
)

display(friends_df)

,Strategy,Its friend
0,(2):Always_Betray,(3):Always_Cooperate
1,(4):Patient_Unforgiving (patience=1),(5):Copycat (1st:Cooperate)
2,(1):Random_Strategy (p_coop=0.1),(3):Always_Cooperate
3,(5):Copycat (1st:Cooperate),(4):Patient_Unforgiving (patience=1)
4,(6):Periodic (period=1),(3):Always_Cooperate
5,(0):Random_Strategy (p_coop=0.5),(3):Always_Cooperate
6,(3):Always_Cooperate,(4):Patient_Unforgiving (patience=1)


## Dataframes

In [13]:
average_reward_per_round_df = pd.DataFrame.from_dict(average_rewards_per_round, orient="index", columns=["Average Reward"])
average_reward_per_round_df.index.name = "Strategy Name"
average_reward_per_round_df = average_reward_per_round_df.round(3)
average_reward_per_round_df

,Average Reward
Strategy Name,
(2):Always_Betray,2.061
(4):Patient_Unforgiving (patience=1),2.056
(1):Random_Strategy (p_coop=0.1),1.940
(5):Copycat (1st:Cooperate),1.869
(6):Periodic (period=1),1.503
(0):Random_Strategy (p_coop=0.5),1.496
(3):Always_Cooperate,1.327


In [14]:
# Per match

average_reward_per_match_df = pd.DataFrame.from_dict(average_rewards_per_match, orient="index", columns=["Average Reward"])
average_reward_per_match_df.index.name = "Strategy Name"
average_reward_per_match_df = average_reward_per_match_df.round(3)
average_reward_per_match_df

,Average Reward
Strategy Name,
(2):Always_Betray,2061.143
(4):Patient_Unforgiving (patience=1),2056.286
(1):Random_Strategy (p_coop=0.1),1939.714
(5):Copycat (1st:Cooperate),1869.429
(6):Periodic (period=1),1502.714
(0):Random_Strategy (p_coop=0.5),1496.429
(3):Always_Cooperate,1327.286


In [15]:
strat_names = [str(strategies[s]) for s in total_rewards.keys()]

score_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    score_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2], scores[s1])
    score_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s1], scores[s2])

for s in strat_names:
    score_matrix.loc[s, s] = (0, 0)

display(score_matrix)

,(2):Always_Betray,(4):Patient_Unforgiving (patience=1),(1):Random_Strategy (p_coop=0.1),(5):Copycat (1st:Cooperate),(6):Periodic (period=1),(0):Random_Strategy (p_coop=0.5),(3):Always_Cooperate
(2):Always_Betray,"(0, 0)","(1004, 999)","(1416, 896)","(1004, 999)","(3000, 500)","(3004, 499)","(5000, 0)"
(4):Patient_Unforgiving (patience=1),"(999, 1004)","(0, 0)","(1355, 915)","(3000, 3000)","(2997, 507)","(3043, 493)","(3000, 3000)"
(1):Random_Strategy (p_coop=0.1),"(896, 1416)","(915, 1355)","(0, 0)","(1283, 1278)","(2868, 813)","(2828, 888)","(4788, 318)"
(5):Copycat (1st:Cooperate),"(999, 1004)","(3000, 3000)","(1278, 1283)","(0, 0)","(2498, 2503)","(2311, 2316)","(3000, 3000)"
(6):Periodic (period=1),"(500, 3000)","(507, 2997)","(813, 2868)","(2503, 2498)","(0, 0)","(2196, 2261)","(4000, 1500)"
(0):Random_Strategy (p_coop=0.5),"(499, 3004)","(493, 3043)","(888, 2828)","(2316, 2311)","(2261, 2196)","(0, 0)","(4018, 1473)"
(3):Always_Cooperate,"(0, 5000)","(3000, 3000)","(318, 4788)","(3000, 3000)","(1500, 4000)","(1473, 4018)","(0, 0)"


In [16]:
sum_matrix = pd.DataFrame(index=strat_names, columns=strat_names, dtype=object)

for (s1, s2), scores in duel_rewards.items():
    sum_matrix.loc[str(strategies[s2]), str(strategies[s1])] = (scores[s2] + scores[s1])
    sum_matrix.loc[str(strategies[s1]), str(strategies[s2])] = (scores[s2] + scores[s1])

for s in strat_names:
    sum_matrix.loc[s, s] = 0

display(sum_matrix)

,(2):Always_Betray,(4):Patient_Unforgiving (patience=1),(1):Random_Strategy (p_coop=0.1),(5):Copycat (1st:Cooperate),(6):Periodic (period=1),(0):Random_Strategy (p_coop=0.5),(3):Always_Cooperate
(2):Always_Betray,0,2003,2312,2003,3500,3503,5000
(4):Patient_Unforgiving (patience=1),2003,0,2270,6000,3504,3536,6000
(1):Random_Strategy (p_coop=0.1),2312,2270,0,2561,3681,3716,5106
(5):Copycat (1st:Cooperate),2003,6000,2561,0,5001,4627,6000
(6):Periodic (period=1),3500,3504,3681,5001,0,4457,5500
(0):Random_Strategy (p_coop=0.5),3503,3536,3716,4627,4457,0,5491
(3):Always_Cooperate,5000,6000,5106,6000,5500,5491,0


In [17]:
victory_matrix = score_matrix.apply(lambda col: col.map(lambda x: int(x[0] > x[1])))
for s in strat_names:
    victory_matrix.loc[s, s] = float("NaN")

display(victory_matrix)

,(2):Always_Betray,(4):Patient_Unforgiving (patience=1),(1):Random_Strategy (p_coop=0.1),(5):Copycat (1st:Cooperate),(6):Periodic (period=1),(0):Random_Strategy (p_coop=0.5),(3):Always_Cooperate
(2):Always_Betray,NaN,1.0,1.0,1.0,1.0,1.0,1.0
(4):Patient_Unforgiving (patience=1),0.0,NaN,1.0,0.0,1.0,1.0,0.0
(1):Random_Strategy (p_coop=0.1),0.0,0.0,NaN,1.0,1.0,1.0,1.0
(5):Copycat (1st:Cooperate),0.0,0.0,0.0,NaN,0.0,0.0,0.0
(6):Periodic (period=1),0.0,0.0,0.0,1.0,NaN,0.0,1.0
(0):Random_Strategy (p_coop=0.5),0.0,0.0,0.0,1.0,1.0,NaN,1.0
(3):Always_Cooperate,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [18]:
def color_cell(x):
    if not isinstance(x, tuple):
        return ""
    if x[0] == 0 and x[1] == 0:
        return "background-color: black"
    elif x[0] > x[1]:
        return "background-color: green"
    elif x[0] < x[1]:
        return "background-color: darkred"
    else:
        return "background-color: gray"

styled_score_matrix = (
    score_matrix.style
    .map(color_cell)
    .set_properties(**{
        "border": "1px solid black"
    })
    .set_table_styles([
        {"selector": "th", "props": [("border", "2px solid black")]},
        {"selector": "td", "props": [("border", "2px solid black")]}
    ])
)

display(styled_score_matrix)

,(2):Always_Betray,(4):Patient_Unforgiving (patience=1),(1):Random_Strategy (p_coop=0.1),(5):Copycat (1st:Cooperate),(6):Periodic (period=1),(0):Random_Strategy (p_coop=0.5),(3):Always_Cooperate
(2):Always_Betray,"(0, 0)","(1004, 999)","(1416, 896)","(1004, 999)","(3000, 500)","(3004, 499)","(5000, 0)"
(4):Patient_Unforgiving (patience=1),"(999, 1004)","(0, 0)","(1355, 915)","(3000, 3000)","(2997, 507)","(3043, 493)","(3000, 3000)"
(1):Random_Strategy (p_coop=0.1),"(896, 1416)","(915, 1355)","(0, 0)","(1283, 1278)","(2868, 813)","(2828, 888)","(4788, 318)"
(5):Copycat (1st:Cooperate),"(999, 1004)","(3000, 3000)","(1278, 1283)","(0, 0)","(2498, 2503)","(2311, 2316)","(3000, 3000)"
(6):Periodic (period=1),"(500, 3000)","(507, 2997)","(813, 2868)","(2503, 2498)","(0, 0)","(2196, 2261)","(4000, 1500)"
(0):Random_Strategy (p_coop=0.5),"(499, 3004)","(493, 3043)","(888, 2828)","(2316, 2311)","(2261, 2196)","(0, 0)","(4018, 1473)"
(3):Always_Cooperate,"(0, 5000)","(3000, 3000)","(318, 4788)","(3000, 3000)","(1500, 4000)","(1473, 4018)","(0, 0)"


In [19]:
import webbrowser
store_styled_matrix_in_html = False
if store_styled_matrix_in_html:
    styled_score_matrix.to_html("styled_score_matrix.html")
    webbrowser.open_new_tab("styled_score_matrix.html")

## Wyniki

In [34]:
def get_name(id : int) -> str:
    return "".join(str(strategies[id]).split(":")[1:])

summary = {}
for ix, (id, score) in enumerate(total_rewards.items()):
    summary[ix] = {"Name": get_name(id),
                   "Nemesis": get_name(nemesis[id][0]),
                   "Friend": get_name(friends[id][0])}

summary_df = pd.DataFrame.from_dict(summary, orient="index")
summary_df.index.name = "Place"

display(summary_df)

,Name,Nemesis,Friend
Place,,,
0,Always_Betray,Patient_Unforgiving (patience=1),Always_Cooperate
1,Patient_Unforgiving (patience=1),Always_Betray,Copycat (1stCooperate)
2,Random_Strategy (p_coop=0.1),Always_Betray,Always_Cooperate
3,Copycat (1stCooperate),Always_Betray,Patient_Unforgiving (patience=1)
4,Periodic (period=1),Always_Betray,Always_Cooperate
5,Random_Strategy (p_coop=0.5),Patient_Unforgiving (patience=1),Always_Cooperate
6,Always_Cooperate,Always_Betray,Patient_Unforgiving (patience=1)


dupajaś
